# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (code only)

Reads the gated warehouse through DuckDB. The token comes from a Colab Secret named `HF_TOKEN`, never from a cell.

In [ ]:
%pip -q install duckdb scikit-learn pyarrow

In [ ]:
import os, getpass, json
import duckdb, numpy as np, pandas as pd

# Token order: env var -> Colab Secret -> prompt (last resort). Never paste a token into a cell.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

# One mid-panel month. June 2026 (and the _sample table) stay untouched: sealed test month.
MONTH       = '2026-03'
M_START     = '2026-03-01'   # first day of the feature window
EARLY_END   = '2026-03-08'   # inside the feature window: days 1-7 vs days 8-15
DECISION    = '2026-03-16'   # the decision moment: everything before this is knowable
M_END       = '2026-04-01'   # exclusive end of the outcome window
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"
print('slice:', MONTH, '| decision moment:', DECISION)

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**The contract, in five plain answers**

1. **What one row means.** In the warehouse table, one row is one content item (a pseudonymized page) on one client's site on one day. In my feature frame, one row is one content item over the first half of March 2026.
2. **Which table.** `fact_content_daily_performance`, only the `month=2026-03` partition. I do not use the `_sample` table, which is exactly June 2026.
3. **Which time window.** Features come from 1-15 March 2026. The decision moment is the end of 15 March. The outcome window is 16-31 March. June 2026 stays sealed as the final test month.
4. **What I rank, and the label.** Lane: Refresh / Content Opportunity Scoring. I rank pages by the chance that their daily-average search impressions in 16-31 March fall more than 20% below their daily average in 1-15 March, so a person reviews the likeliest decliners first. This label is a proxy for "worth a look." It does not show why a page declined or that an edit would fix it.
5. **One thing I deliberately exclude.** The GA4 (analytics) columns. Rows before a client's GA4 start are zero-filled with `ga4_data_available = FALSE`, so a zero there means "not tracked yet," not "no visitors." Query 3 below counts how many rows survive when I filter with `IS TRUE`.

These are claims to check, not results. The queries below confirm or contradict them.

In [ ]:
# Code cell: the contract as a small table, so the notebook shows it in one place.
contract = pd.DataFrame({
    'question': ['row means', 'table', 'time window', 'rank / label', 'deliberately excluded'],
    'answer': [
        'content item x client x day (raw); content item over 1-15 Mar (feature frame)',
        'fact_content_daily_performance, month=2026-03 only',
        f'features {M_START}..{DECISION} (exclusive); outcome {DECISION}..{M_END} (exclusive); June 2026 sealed',
        'daily-avg impressions in 2nd half fall >20% vs 1st half (proxy); ranked review queue',
        'GA4 columns (zero-filled before ga4_data_start; see query 3)',
    ],
})
pd.set_option('display.max_colwidth', None)
contract

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Fields | Why |
|---|---|---|
| **Feature** | `imp_h1`, `avg_pos_h1`, `ctr_h1`, `active_days_h1`, `trend_ratio_h1` (all built from 1-15 March only) | Each is knowable at the decision moment (see section 3) |
| **Label / proxy** | `gsc_impressions` summed over 16-31 March (`imp_h2`), turned into `label` | This is the outcome. It never becomes a feature |
| **Context** | `report_date`, `client_hash_id`, `content_hash_id` | Grouping, joining and splitting only. `client_hash_id` defines the train/test split |
| **Excluded** | `ga4_data_available` and every GA4 column | Zero-filled before a client's GA4 start, so zeros do not mean "no engagement" |
| **Excluded** | `sessions_ai` | Very sparse in this release, so a model would learn noise |
| **Excluded** | June 2026 and the `_sample` table | Sealed test month. It is the natural outcome window of any past-to-future label |

In [ ]:
# Code cell: the same sort, as a dictionary the later cells read from.
FEATURES = ['imp_h1', 'avg_pos_h1', 'ctr_h1', 'active_days_h1', 'trend_ratio_h1']
BUCKETS = {
    'feature': FEATURES,
    'label_or_proxy': ['imp_h2 -> label'],
    'context': ['report_date', 'client_hash_id', 'content_hash_id'],
    'excluded': ['ga4_data_available + GA4 columns', 'sessions_ai', 'June 2026 / _sample'],
}
for k, v in BUCKETS.items():
    print(f'{k:15}', v)

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Exactly three verification queries, in order: **grain**, **row count and date span**, **availability**. After them come the five features and the leakage trap.

### Query 1 of 3: grain

Claim: one row is one `report_date x client x content`. If that is true, grouping by those three columns and keeping groups with more than one row returns nothing.

In [ ]:
# Query 1: grain probe. Zero rows back means the grain holds.
grain = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {FACT}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print('duplicate-grain rows found:', len(grain))
grain

### Query 2 of 3: my slice's row count and date span

Claim: the slice covers 1-31 March 2026. The query counts the rows and checks the first and last day.

In [ ]:
# Query 2: row count + date span of the slice.
span = con.sql(f"""
    SELECT COUNT(*)                        AS n_rows,
           MIN(report_date)                AS first_day,
           MAX(report_date)                AS last_day,
           COUNT(DISTINCT client_hash_id)  AS clients,
           COUNT(DISTINCT content_hash_id) AS content_items
    FROM {FACT}
""").df()
span

### Query 3 of 3: availability (`IS TRUE`)

Claim: GA4 columns are only trustworthy where `ga4_data_available IS TRUE`. The query shows how many rows survive that filter. `IS TRUE` matters because a plain `= TRUE` drops NULLs without saying so, and a NULL flag is not the same as "available".

In [ ]:
# Query 3: availability. How many rows survive ga4_data_available IS TRUE?
avail = con.sql(f"""
    SELECT COUNT(*)                                                      AS total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)            AS rows_surviving_is_true,
           ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)
                 / COUNT(*), 2)                                          AS pct_surviving
    FROM {FACT}
""").df()
avail

### The five features

Not a verification query: this is the feature build. One SQL aggregation per content item, from 1-15 March only, then a small pandas frame. Pages with fewer than 100 impressions in the feature window are dropped, because tiny counts make a "decline" meaningless.

| Feature | Knowable at the decision moment because... |
|---|---|
| `imp_h1` | it is the sum of impressions on 1-15 March, all of which are in the past on 16 March |
| `avg_pos_h1` | it is the average search position over days 1-15 (days with impressions only), all observed before 16 March |
| `ctr_h1` | it is clicks divided by impressions on days 1-15, both observed before 16 March |
| `active_days_h1` | it is the count of days 1-15 with any impressions, observed before 16 March |
| `trend_ratio_h1` | it compares impressions on days 8-15 with days 1-7, both inside the feature window, so it uses no day on or after 16 March |

In [ ]:
# Feature build + label. Features use report_date < DECISION; the label uses report_date >= DECISION.
frame = con.sql(f"""
    WITH m AS (
        SELECT * FROM {FACT}
        WHERE report_date >= DATE '{M_START}' AND report_date < DATE '{M_END}'
    ),
    h1 AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions)                                   AS imp_h1,
               SUM(gsc_clicks)                                        AS clicks_h1,
               AVG(gsc_avg_position) FILTER (WHERE gsc_impressions > 0) AS avg_pos_h1,
               COUNT(*) FILTER (WHERE gsc_impressions > 0)            AS active_days_h1,
               SUM(CASE WHEN report_date >= DATE '{EARLY_END}' THEN gsc_impressions ELSE 0 END) AS imp_late_h1,
               SUM(CASE WHEN report_date <  DATE '{EARLY_END}' THEN gsc_impressions ELSE 0 END) AS imp_early_h1
        FROM m
        WHERE report_date < DATE '{DECISION}'
        GROUP BY 1, 2
    ),
    h2 AS (
        SELECT client_hash_id, content_hash_id,
               COALESCE(SUM(gsc_impressions), 0) AS imp_h2
        FROM m
        WHERE report_date >= DATE '{DECISION}'
        GROUP BY 1, 2
    )
    SELECT h1.*, COALESCE(h2.imp_h2, 0) AS imp_h2
    FROM h1 LEFT JOIN h2 USING (client_hash_id, content_hash_id)
    WHERE h1.imp_h1 >= 100
""").df()

n_before = len(frame)
frame['ctr_h1'] = frame['clicks_h1'] / frame['imp_h1']
frame['trend_ratio_h1'] = (frame['imp_late_h1'] + 1) / (frame['imp_early_h1'] + 1)

# Label (proxy): daily-average impressions in 16-31 Mar (16 days) fell >20% vs 1-15 Mar (15 days).
frame['label'] = ((frame['imp_h2'] / 16) < 0.8 * (frame['imp_h1'] / 15)).astype(int)

df = frame.dropna(subset=FEATURES).reset_index(drop=True)
print(f'pages with >=100 impressions in 1-15 Mar: {n_before:,}')
print(f'dropped for a missing feature value:      {n_before - len(df):,}')
print(f'rows in the modelling frame:              {len(df):,}')
print(f'share labelled "declining" (base rate):   {df.label.mean():.3f}')
df[['client_hash_id', 'content_hash_id'] + FEATURES + ['label']].head()

### The trap: one label-derived column, on purpose

`drop_ratio_leak` is the outcome-window daily average divided by the feature-window daily average. That is the label's own formula. It cannot be known on 16 March. I add it, watch the quick score jump toward perfect, then delete it and keep the honest number.

Quick score = ROC AUC (1.0 is perfect, 0.5 is a coin flip) from a small random forest, tested on whole clients it never saw. The plain rule ranks pages by the lowest `trend_ratio_h1` and gets the same test.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

def quick_score(frame, cols, seed=42):
    # Whole clients held out: no client appears in both train and test.
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed)
    tr, te = next(gss.split(frame, frame['label'], groups=frame['client_hash_id']))
    rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=seed, n_jobs=-1)
    rf.fit(frame.iloc[tr][cols], frame['label'].iloc[tr])
    auc = roc_auc_score(frame['label'].iloc[te], rf.predict_proba(frame.iloc[te][cols])[:, 1])
    rule = roc_auc_score(frame['label'].iloc[te], -frame['trend_ratio_h1'].iloc[te])  # plain rule
    return auc, rule, len(te)

# 1) Add the leak on purpose.
df['drop_ratio_leak'] = (df['imp_h2'] / 16) / (df['imp_h1'] / 15)
leak_auc, _, n_test = quick_score(df, FEATURES + ['drop_ratio_leak'])
print(f'WITH the leaked column  : ROC AUC = {leak_auc:.3f}   (test rows: {n_test:,})')

# 2) Delete it. Keep the honest number.
df = df.drop(columns=['drop_ratio_leak'])
assert 'drop_ratio_leak' not in df.columns
honest_auc, rule_auc, n_test = quick_score(df, FEATURES)
print(f'WITHOUT it (honest)     : ROC AUC = {honest_auc:.3f}')
print(f'Plain rule (low trend)  : ROC AUC = {rule_auc:.3f}')

os.makedirs('work/outputs', exist_ok=True)
with open('work/outputs/w03_scores.json', 'w') as f:
    json.dump({'slice': MONTH, 'rows': int(len(df)), 'base_rate': float(df.label.mean()),
               'auc_with_leak': float(leak_auc), 'auc_honest': float(honest_auc),
               'auc_plain_rule': float(rule_auc), 'seed': 42}, f, indent=2)

**How to read it.** The leaked score is meaningless: the model was handed the answer. The honest score is the one I keep. Read it next to the base rate and the plain rule above, and say which is higher. The leaked column is deleted; the code asserts it is gone.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation: one month and an unbalanced panel.** My slice is one month (March 2026) split into two short windows, and clients have very different amounts of history and traffic. A few clients may supply most of the rows, so the honest score can mostly describe those clients. A decline between the first and second half of one month can also be a weekday pattern or normal noise, not a lasting decline. The cell below measures how concentrated the rows are, and whether the score changes with a different seed.

In [ ]:
# Code cell: how concentrated is the frame, and does the honest score move with the seed?
share = df['client_hash_id'].value_counts(normalize=True)
print(f'clients in the frame: {len(share)}')
print(f'top 3 clients supply {share.head(3).sum():.1%} of rows')
for s in (1, 7, 42):
    a, r, _ = quick_score(df, FEATURES, seed=s)
    print(f'seed {s:>2}: model AUC {a:.3f} | plain rule AUC {r:.3f}')

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.